In [1]:
# DO NOT CONTAINERISE
# =====
# Dependency
# -----
# install.packages("\")
# # install.packages("dplyr")
# # install.packages("tidyr")

# install.packages("vegan")

library(vegan)
# tidyverse
library(dplyr)
library(tidyr)

# base settings
# -----
conf_vlab_name     <- "DNA"
# conf_workflow_name <- "PEMA"

# conf_workflow_id   <- f"wid-{datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
param_workflow_name <- "workflow name"

# dev
# -----
# library: --volume="//c/DockerShare/DNA:/home/jovyan" naavre-fl-dna-jupyter:local
# NaaVRE: /home/jovyan/Virtual Labs/DNA/Git public
# dir_code <- file.path("/", "home", "jovyan", "Virtual Labs", conf_vlab_name, "Git public", "library")
# if (!dir.exists(dir_code)) {dir.create(dir_code, recursive=TRUE)}

# dir_data <- file.path("/", "home", "jovyan", "Cloud Storage", "naa-vre-user-data", conf_vlab_name, param_workflow_name)
# if (!dir.exists(dir_data)) {dir.create(dir_data, recursive=TRUE)}

# local
# -----
conf_dir_workspace <- file.path("/", "home", "jovyan", "Cloud Storage")

conf_dir_data_local_tmp <- file.path("/", "tmp", "data")

# MINIO
# -----
conf_minio_public_bucket      <- "naa-vre-public"
conf_minio_public_bucket_root <- paste("vl", tolower(conf_vlab_name), sep="-")
conf_minio_public_local_root  <- file.path(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root)
conf_minio_public_local_code  <- file.path(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "code")
conf_minio_public_local_data  <- file.path(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "data")

conf_minio_user_bucket        <- "naa-vre-user-data"
# conf_minio_user_bucket_root   <- param_user_email
conf_minio_user_bucket_root   <- conf_vlab_name
conf_minio_user_local_root    <- file.path(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root)
conf_minio_user_local_code    <- file.path(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   "library")
conf_minio_user_local_data    <- file.path(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   param_workflow_name)
conf_minio_user_local_flog    <- file.path(conf_minio_user_local_data, "log.md")

# API key
# -----
# If running under NaaVRE, input `your api key` with the correct value and input in the GUI:
# secret_SERVICE_KEY = "d18e08911c964d45912eb1e954adf994"
# secret_SERVICE_KEY = SecretsProvider().set_secret("secret_SERVICE_KEY")
# secret_SERVICE_KEY = SecretsProvider().get_secret("secret_SERVICE_KEY")

# for workflow step
# .....
# if os.path.exists(conf_minio_user_local_flog):
#     with open(conf_minio_user_local_flog, "a+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 
# else:
#     if not os.path.exists(conf_minio_user_local_data):
#         os.makedirs(conf_minio_user_local_data)
#     with open(conf_minio_user_local_flog, "w+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 

# Input param
# -----
# PEMA-SequenceRetriever
# .....
param_gene_sequences <- "SRR3231901"

# OTU
# .....
# pema_otu_delimiter = "\t"
# bold_otu_delimiter = ","
conf_delimiter_tsv <- "\t"
conf_delimiter_csv <- ","

print("Finish: NaaVRE parameters")
print("Workspace public:")
print(paste("  Root", conf_minio_public_local_root, sep=": "))
print(paste("  Code", conf_minio_public_local_code, sep=": "))
print(paste("  Data", conf_minio_public_local_data, sep=": "))

print("Workspace user:")
print(paste("  Root", conf_minio_user_local_root, sep=": "))
print(paste("  Code", conf_minio_user_local_code, sep=": "))
print(paste("  Data", conf_minio_user_local_data, sep=": "))
print(paste("  Log",  conf_minio_user_local_flog, sep=":  "))


Loading required package: permute


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




[1] "Finish: NaaVRE parameters"
[1] "Workspace public:"
[1] "  Root: //home/jovyan/Cloud Storage/naa-vre-public/vl-dna"
[1] "  Code: //home/jovyan/Cloud Storage/naa-vre-public/vl-dna/code"
[1] "  Data: //home/jovyan/Cloud Storage/naa-vre-public/vl-dna/data"
[1] "Workspace user:"
[1] "  Root: //home/jovyan/Cloud Storage/naa-vre-user-data/DNA"
[1] "  Code: //home/jovyan/Cloud Storage/naa-vre-user-data/DNA/library"
[1] "  Data: //home/jovyan/Cloud Storage/naa-vre-user-data/DNA/workflow name"
[1] "  Log:  //home/jovyan/Cloud Storage/naa-vre-user-data/DNA/workflow name/log.md"


In [2]:
# DNA, workflow start
# ---
# NaaVRE:
#  cell:
#   outputs:
#    - dummy_cell_arg_o: String
# ...

# library(vegan)

# prepare folders
# .....
if (!dir.exists(conf_dir_data_local_tmp)) {dir.create(conf_dir_data_local_tmp, recursive=TRUE)}

# if (!dir.exists(conf_minio_public_local_root)) {dir.create(conf_minio_public_local_root, recursive=TRUE)}

if (!dir.exists(conf_minio_user_local_root)) {dir.create(conf_minio_user_local_root, recursive=TRUE)}

if (!dir.exists(conf_minio_user_local_data)) {dir.create(conf_minio_user_local_data, recursive=TRUE)}

fp_log <- file(conf_minio_user_local_flog)
writeLines(paste("# ", param_workflow_name, "\n", sep=""), fp_log)
close(fp_log)

# create log
# .....
print(param_workflow_name)
workflow_step <- paste(conf_vlab_name, "-Start", sep="")

if (file.exists(conf_minio_user_local_flog)) {
    cat(paste("\n## ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")
} else {
    if (!dir.exists(conf_minio_user_local_data)) {dir.create(conf_minio_user_local_data, recursive=TRUE)}
    cat(paste("\n## ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=FALSE, sep="\n")
}

# lib, minio_public
# -----

# lib, minio_user
# -----

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

# func
# -----

# start
# -----

# finish
# -----
cat(paste("\nFinish: ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")
cat(paste("\nOutput: ", conf_minio_user_local_data, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")

print(paste("Finish: ", workflow_step, sep=""))


[1] "workflow name"
[1] "Finish: DNA-Start"


In [3]:
# DNA, PEMA-Converter
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

library(vegan)
# tidyverse
library(dplyr)
library(tidyr)

print(param_workflow_name)
workflow_step_i <- paste(conf_vlab_name, "-OTU_Unify", sep="")
workflow_step   <- paste(conf_vlab_name, "-PEMA_Converter", sep="")

if (file.exists(conf_minio_user_local_flog)) {
    cat(paste("\n## ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")
} else {
    if (!dir.exists(conf_minio_user_local_data)) {dir.create(conf_minio_user_local_data, recursive=TRUE)}
    cat(paste("\n## ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=FALSE, sep="\n")
}

# lib, minio_public
# -----

# lib, minio_user
# -----

# input
# -----
dummy_cell_arg_i <- "dummy input"

dir_otu <- file.path(conf_minio_user_local_data, workflow_step_i)
fname_otu_i <- 'otu.tsv'
file_otu_i  <- file.path(dir_otu, fname_otu_i)
# print(paste("DevOps: file_otu_i", file_otu_i, sep=": "))

# now resume creating the aggregation file
# 域”、“门”、“纲”、“目”、“科”、“属”、“种”
colnames    <- c("Domain", "Phylum", "Class", "Order", "Family", "Genus", "Species")
new_colname <- "OTU" 

col_order <- c(new_colname, "Species", "Genus", "Family", "Order", "Class", "Phylum", "Domain")

num_tries     = 0
previous_nans = -1

# output
# -----
dummy_cell_arg_o <- "dummy output"

dir_pema <- file.path(conf_minio_user_local_data, workflow_step)
if (!dir.exists(dir_pema)) {dir.create(dir_pema, recursive=TRUE)}

# write.csv
fname_abundance_o          <- 'abundance.csv'
file_abundance_o           <- file.path(dir_pema, fname_abundance_o)
fname_aggregation_o        <- 'aggregation.csv'
file_aggregation_o         <- file.path(dir_pema, fname_aggregation_o)
fname_abundance_no_dupls_o <- 'abundance_mds.csv'
file_abundance_no_dupls_o  <- file.path(dir_pema, fname_abundance_no_dupls_o)

# func
# -----

# start
# -----
# import the biotic data, i.e. the final_table.tsv as it derives from PEMA
df_abundance <- read.csv(file_otu_i, sep=conf_delimiter_tsv, header=TRUE) 

# now we have an abundance table
# in the 1st column we have an OTU/ASV code
# then we have several columns, one for each sample
# and last, we have the classification column, with the taxonomy of each OTU/ASV

# start creating the aggregation file, needed for certain function (e.g. tax2dist)
df_taxonomy <- select(df_abundance, classification)
# print("DevOps: df_taxonomy-00")
# print(df_taxonomy)

# remove classification column from biotic data
df_abundance <- select(df_abundance, -classification)
# print("DevOps: df_abundance-00")
# print(df_abundance)

# save the biotic data, i.e. the community data that are needed as an input to 
# most of the RvLab functions
write.csv(df_abundance, file_abundance_o, quote=F, row.names=FALSE)

df_taxonomy <- separate(df_taxonomy, classification, colnames, sep=";", remove=TRUE, convert=FALSE, extra="warn", fill="warn")
df_taxonomy <- cbind(df_abundance$OTU, df_taxonomy)
# print("DevOps: df_taxonomy-01")
# print(df_taxonomy)

names(df_taxonomy)[names(df_taxonomy) == "df_abundance$OTU"] <- new_colname
# print("DevOps: df_taxonomy-02")
# print(df_taxonomy)

# check where there are NA values in the taxonomy data frame
colSums(is.na(df_taxonomy))

repeat {
    for (i in 1:nrow(df_taxonomy))
        if (is.na(df_taxonomy$Species[i])){
            df_taxonomy$Species[i] = df_taxonomy$Genus[i]
            df_taxonomy$Genus[i]   = df_taxonomy$Family[i]
            df_taxonomy$Family[i]  = df_taxonomy$Order[i]
            df_taxonomy$Order[i]   = df_taxonomy$Class[i]
            df_taxonomy$Class[i]   = df_taxonomy$Phylum[i]
    }

    current_nans = sum(is.na(df_taxonomy$Species))
    if (current_nans==0) {
        break
    }

    num_tries = num_tries + 1
    if (num_tries == 1000 && previous_nans == current_nans){
        stop("It seems that there are rows which cannot be fixed. Please check your data and contact the LifeWatch ERIC team for help.")
    }

    previous_nans = current_nans
}

# change the order of the columns
df_taxonomy_ordered <- df_taxonomy[, col_order]

# Remove duplicates based on Species column
df_taxonomy_ordered_no_dupls <- df_taxonomy_ordered[!duplicated(df_taxonomy_ordered$Species), ]

# save the aggregation file
write.csv(df_taxonomy_ordered_no_dupls, file_aggregation_o, quote=F, row.names=FALSE)

# create an abundance table that has the same rows as the aggregation file
df_abundance_no_dupls <- df_abundance %>% filter(OTU %in% df_taxonomy_ordered_no_dupls$OTU)

# save it
write.csv(df_abundance_no_dupls, file_abundance_no_dupls_o, quote=F, row.names=FALSE)

# finish
# -----
cat(paste("\nFinish: ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")
cat(paste("\nOutput: ", dir_pema, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")

print(paste("Finish: ", workflow_step, sep=""))


[1] "workflow name"


Warning message:
“Expected 7 pieces. Additional pieces discarded in 4 rows [1, 4, 6, 7].”
Warning message:
“Expected 7 pieces. Missing pieces filled with `NA` in 4 rows [2, 3, 5, 8].”


OTU  Domain  Phylum   Class   Order  Family   Genus Species 
      0       0       0       0       0       0       2       4

[1] "Finish: DNA-PEMA_Converter"


In [4]:
# DNA, metamds-observations
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
#   outputs:
#    - dummy_cell_arg_o: String
# ...

library(vegan)
# tidyverse
library(dplyr)
library(tidyr)

print(param_workflow_name)
workflow_step_i <- paste(conf_vlab_name, "-PEMA_Converter", sep="")
workflow_step   <- paste(conf_vlab_name, "-metamds-observations", sep="")

if (file.exists(conf_minio_user_local_flog)) {
    cat(paste("\n## ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")
} else {
    if (!dir.exists(conf_minio_user_local_data)) {dir.create(conf_minio_user_local_data, recursive=TRUE)}
    cat(paste("\n## ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=FALSE, sep="\n")
}

# lib, minio_public
# -----

# lib, minio_user
# -----

# input
# -----
dummy_cell_arg_i <- "dummy input"

dir_pema <- file.path(conf_minio_user_local_data, workflow_step_i)
# print(paste("DevOps: dir_pema", dir_pema, sep=": "))

# -- args[1]: community matrix data file, i.e. abundance file
# fname_abundance = "matrix_from_gitlab-dev.csv"
# fname_abundance = "abundance.csv"
fname_abundance <- "abundance_mds.csv"
file_abundance  <- file.path(dir_pema, fname_abundance)

# -- args[2]: TRUE | FALSE (transpose matrix)
#   Note: dimension of df_matrix from file_abundance
#     Ture:  ncols >= 3
#     FALSE: nrows >= 3
is_transpose_abundance <- FALSE
# is_transpose_abundance <- TRUE

# -- args[3]: (Optional) factor file
#   Note: for mds plot, label & legend names, it's better not to provide
#     the rows should be the same as the names in the index of df_matrix, 
#     caution:
#        is_transpose_abundance
#        df_factor_col_label
# fname_factor_csv <- "factors_manual_from_gitlab-dev.csv"  # require col "AreaType"
fname_factor_csv <- "none"
if (fname_factor_csv != "none") {
    file_factor_csv <- file.path(dir_pema, fname_factor_csv)
} else {
    file_factor_csv <- "none"
}

# -- args[4]: if a factor file was used, the name of the column of the factor file that will be used as a label
# df_factor_col_label <- "LabelName"
df_factor_col_label <- "Location"
# df_factor_col_label <- "AreaType"

# -- args[5]: method for the calculation of distance matrix for the community matrix data file, e.g. euclidean, bray etc. 
mds_distance_matrix <- "bray"

# -- args[6]: TRUE | FALSE (autotransformation)
is_mds_auto_transformation <- FALSE

# -- args[7]: method for the calculation of pairwise distances, e.g. euclidean, bray etc.
mds_distance_pairwise <- "bray"

# -- args[8]: the number of permutations required for the permanova analysis
n_permutations <- 999


# output
# -----
dummy_cell_arg_o <- "dummy output"

dir_metamds <- file.path(conf_minio_user_local_data, workflow_step)
if (!dir.exists(dir_metamds)) {dir.create(dir_metamds, recursive=TRUE)}
# print(paste("DevOps: dir_metamds", dir_metamds, sep=": "))

fname_mds_fig_o <- "mds.png"
file_mds_fig_o  <- file.path(dir_metamds, fname_mds_fig_o)


# func
# -----

# start
# -----
# In order to run this script, you need to have at least 2 samples in your abundance file

# import the community data matrix
df_matrix <- read.csv(file_abundance, sep=conf_delimiter_csv, header=TRUE, row.names=1)
print("DevOps: df_matrix-00")
print(df_matrix)

# replace the empty cells with zeros
df_matrix <-  df_matrix %>% replace(is.na(.), 0)

# import the factor file if it exists
if (file_factor_csv != "none") {
    df_factor <- read.csv(file_factor_csv, sep=conf_delimiter_csv, header=TRUE, row.names=1)
} else {
    # if not, create factors
    df_factor <- colnames(df_matrix)
}
print("DevOps: df_factor-00")
print(df_factor)

# transpose the community data matrix so that taxa would be the variables and samples the rows
if (is_transpose_abundance == TRUE)
    df_matrix <- t(df_matrix)

tmp_matrix_tot <- rowSums(df_matrix)
df_matrix <- df_matrix[tmp_matrix_tot > 0, ]
print("DevOps: df_matrix-01: rowSums(df_matrix) > 0")
print(df_matrix)

# create the mds
print("Info: Create the mds")
if (is_mds_auto_transformation == TRUE){
    sol_nmds <- metaMDS(df_matrix, distance=mds_distance_matrix, autotransform=TRUE)
}else{ 
    sol_nmds <- metaMDS(df_matrix, distance=mds_distance_matrix, autotransform=FALSE)
}

# store the stress value of the mds
print("Info: Store the stress value of the mds")
stress <- paste("Stress:", round(sol_nmds$stress, 2))

# create one label for each level of the chosen factor
if (file_factor_csv != "none") {
    # if(!df_factor_col_label %in% colnames(df_factor)) {
    #     df_factor$df_factor_col_label <- as.vector(rownames(df_factor))
    # }

    labels <- as.factor(df_factor[[df_factor_col_label]])
} else {
    # if not, create factors
    labels <- as.factor(df_factor)
}
# print("DevOps: labels-00")
# print(labels)

# create the mds plot
print("Info: Create the mds plot")

png(file_mds_fig_o, height=800, width=1000, units="px")

op <- par(cex=1.5)
par(xpd=T, mar=par()$mar + c(0, 0, 0, 7))
plot(sol_nmds, type="n", ylab="", xlab="", yaxt="n", xaxt="n", bty="o") 
points(sol_nmds, col=labels, pch=16, cex=1.5)
text(x=1.4, y=2, labels=stress)
if (file_factor_csv != "none") {
    legend("bottomright", legend=unique(df_factor[[df_factor_col_label]]), col=unique(labels), pch=16, bty="n", inset=c(-0.17,0), xpd=T)
} else {
    # if not, create factors
    legend("bottomright", legend=unique(df_factor), col=unique(labels), pch=16, bty="n", inset=c(-0.17,0), xpd=T)
}
par(mar=c(5, 4, 4, 1) + 0.1)
dev.off()

# print the results of the function
print("Info: This is the result of the mds")
print(sol_nmds)

# run the permanova analysis
if (file_factor_csv != "none") {
    permanova <- adonis2(df_matrix ~ df_factor[[df_factor_col_label]], method=mds_distance_pairwise, data=df_factor, permutations=n_permutations)
    print("this is the result of the permanova");
    print(permanova)
} else {
    print("you cannot run the permanova analysis without a factor file")
    # permanova <- adonis2(df_matrix ~ df_factor, method=mds_distance_pairwise, data=df_factor)
    # print("this is the result of the permanova");
    # print(permanova)
}

# finish
# -----
cat(paste("\nFinish: ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")
cat(paste("\nOutput: ", dir_pema, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")

print(paste("Finish: ", workflow_step, sep=""))


[1] "workflow name"
[1] "DevOps: df_matrix-00"
     SRR3231901
Otu8          2
Otu6         13
Otu7          3
Otu4         28
Otu5         18
Otu2         46
Otu3         27
Otu1         88
[1] "DevOps: df_factor-00"
[1] "SRR3231901"
[1] "DevOps: df_matrix-01: rowSums(df_matrix) > 0"
[1]  2 13  3 28 18 46 27 88
[1] "Info: Create the mds"
Run 0 stress 0 
Run 1 stress 0 
... Procrustes: rmse 0.1990401  max resid 0.3280178 
Run 2 stress 0.2848521 
Run 3 stress 8.910312e-05 
... Procrustes: rmse 0.089424  max resid 0.1859187 
Run 4 stress 9.819368e-05 
... Procrustes: rmse 0.03751731  max resid 0.05453469 
Run 5 stress 0.2235651 
Run 6 stress 0 
... Procrustes: rmse 0.2012874  max resid 0.4106685 
Run 7 stress 0.2235651 
Run 8 stress 0 
... Procrustes: rmse 0.1792661  max resid 0.3136711 
Run 9 stress 0 
... Procrustes: rmse 0.1864056  max resid 0.2542685 
Run 10 stress 5.811435e-06 
... Procrustes: rmse 0.02991068  max resid 0.058269 
Run 11 stress 9.767452e-05 
... Procrustes: rmse 0.20

Warning message in metaMDS(df_matrix, distance = mds_distance_matrix, autotransform = FALSE):
“stress is (nearly) zero: you may have insufficient data”


[1] "Info: Store the stress value of the mds"
[1] "Info: Create the mds plot"


species scores not available



agg_record_e8d444f250da 
                      2

[1] "Info: This is the result of the mds"

Call:
metaMDS(comm = df_matrix, distance = mds_distance_matrix, autotransform = FALSE) 

global Multidimensional Scaling using monoMDS

Data:     df_matrix 
Distance: bray 

Dimensions: 2 
Stress:     0 
Stress type 1, weak ties
Best solution was not repeated after 20 tries
The best solution was from try 0 (metric scaling or null solution)
Scaling: centring, PC rotation, halfchange scaling 
Species: scores missing

[1] "you cannot run the permanova analysis without a factor file"
[1] "Finish: DNA-metamds-observations"


In [5]:
# DNA, workflow finish
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - dummy_cell_arg_i: String
# ...

# library(vegan)

# create log
# .....
print(param_workflow_name)
workflow_step <- paste(conf_vlab_name, "-Finish", sep="")

if (file.exists(conf_minio_user_local_flog)) {
    cat(paste("\n## ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")
} else {
    if (!dir.exists(conf_minio_user_local_data)) {dir.create(conf_minio_user_local_data, recursive=TRUE)}
    cat(paste("\n## ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=FALSE, sep="\n")
}

# lib, minio_public
# -----

# lib, minio_user
# -----

# input
# -----
dummy_cell_arg_i <- "dummy input"

# output
# -----
dummy_cell_arg_o <- "dummy output"

# func
# -----

# start
# -----

# finish
# -----
cat(paste("\nFinish: ", workflow_step, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")
cat(paste("\nOutput: ", conf_minio_user_local_data, "\n", sep=""), file=conf_minio_user_local_flog, append=TRUE, sep="\n")

print(paste("Finish: ", workflow_step, sep=""))


[1] "workflow name"
[1] "Finish: DNA-Finish"
